# Case Study: Census

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')


In [ ]:
# polars
import polars as pl
from IPython.display import display
import hvplot.polars


## U.S. National Park Service

Source: [Responsible Datasets in Context](https://www.responsible-datasets-in-context.com/posts/np-data/)

<img src="https://www.nps.gov/common/uploads/structured_data/3C84CF74-1DD8-B71B-0B9C7FF83F7C68EB.jpg?width=990&height=742&mode=crop&quality=78" alt="photo of mountains and trees in Yosemite National Park" width="400 px" />

In [ ]:
# datascience
# load in data
# recreational visits to U.S. national parks in 2024
nps = Table.read_table('US-National-Parks_RecreationVisits_1979-2024.csv')
nps = nps.where("Year", 2024).drop("Year", "Region").sort("Visits", 2024)
nps.show(3)


In [ ]:
# polars
# load in data
# recreational visits to U.S. national parks in 2024
nps_pl = pl.read_csv('US-National-Parks_RecreationVisits_1979-2024.csv')
nps_pl = nps_pl.filter((pl.col("Year") == (2024))).drop("Year", "Region").sort("Visits", descending=True)
nps_pl.head(3)


In [ ]:
# datascience
# `take`
# 1. what is the tenth most visited park?
# 2. how do I see just that row?
# 3. how do I see the row after that?

nps.take(9, 10)


In [ ]:
# polars
# `take`
# 1. what is the tenth most visited park?
# 2. how do I see just that row?
# 3. how do I see the row after that?

nps_pl.gather(9)


In [ ]:
# datascience
# more common: we will use `where` to pick out rows based on some condition
nps.where('State', 'CA').show(2)


In [ ]:
# polars
# more common: we will use `where` to pick out rows based on some condition
nps_pl.filter((pl.col('State') == ('CA'))).head(2)


In [ ]:
# datascience
# ... or equivalently
nps.where('State', are.equal_to('CA')).show(2)


In [ ]:
# polars
# ... or equivalently
nps_pl.filter((pl.col('State') == ('CA'))).head(2)


In [ ]:
# datascience
# what about not equal to?
nps.where('State', are.not_equal_to('CA')).show(2)


In [ ]:
# polars
# what about not equal to?
nps_pl.filter((pl.col('State') != ('CA'))).head(2)


In [ ]:
# datascience
# more common: conditions with numerical values
nps.where('Visits', are.between(1e5, 1e6)).show(2)


In [ ]:
# polars
# more common: conditions with numerical values
nps_pl.filter(pl.col('Visits').is_between(1e5, 1e6, closed='left')).head(2)


In [ ]:
# datascience
# what fraction of parks satisfy the above condition?
nps.where('Visits', are.between(1e5, 1e6)).num_rows / nps.num_rows


In [ ]:
# polars
# what fraction of parks satisfy the above condition?
nps_pl.filter(pl.col('Visits').is_between(1e5, 1e6, closed='left')).height / nps_pl.height


### Discussion Questions

The table `nps` has columns:


| Park | Park Code | State | Visits |
| :-- | :-- | :-- | :-- |
| ... | ... | ... | ... |


#### a) Create an **array** containing the names of all California parks that had more than 1 million visitors.

In [ ]:
# ... code here ...


#### b) After evaluating these two expressions in order, what's the result of the second one?

In [ ]:
# datascience
nps.drop('State')
nps.num_columns


In [ ]:
# polars
nps_pl.drop('State')
nps_pl.width


<br/><br/><br/>

---

## Census ##

In [ ]:
# datascience
full = Table.read_table('nc-est2025-agesex-res.csv')
full.show(5)


In [ ]:
# polars
full_pl = pl.read_csv('nc-est2025-agesex-res.csv')
full_pl.head(5)


In [ ]:
# datascience
partial = full.select('SEX', 'AGE', 'POPESTIMATE2020', 'POPESTIMATE2025')
partial.show(6)


In [ ]:
# polars
partial_pl = full_pl.select('SEX', 'AGE', 'POPESTIMATE2020', 'POPESTIMATE2025')
partial_pl.head(6)


### Newborns

**DISCUSS**: How can there be different values for newborn babies vs. five year olds five years later?

In [ ]:
# datascience
partial.column(3).take(5) / partial.column(2).take(0)


In [ ]:
# polars
partial_pl.to_series(3).gather(5) / partial_pl.to_series(2).gather(0)


In [ ]:
# datascience
# make column labels a bit nicer
us_pop = partial.relabeled(2, '2020').relabeled(3, '2025')
us_pop.show(5)


In [ ]:
# polars
# make column labels a bit nicer
us_pop_pl = partial_pl.rename({partial_pl.columns[2]: '2020'}).rename({partial_pl.rename({partial_pl.columns[2]: '2020'}).columns[3]: '2025'})
us_pop_pl.head(5)


### Elderly

In [ ]:
# datascience
us_pop.where('AGE', are.above_or_equal_to(98)).sort('AGE').show()


In [ ]:
# polars
with pl.Config(tbl_rows=-1):
    display(us_pop_pl.filter((pl.col('AGE') >= (98))).sort('AGE'))


In [ ]:
# datascience
us_pop.where('SEX', 0).where('AGE', are.above_or_equal_to(98)).sort('AGE').show()


In [ ]:
# polars
with pl.Config(tbl_rows=-1):
    display(us_pop_pl.filter((pl.col('SEX') == (0))).filter((pl.col('AGE') >= (98))).sort('AGE'))


In [ ]:
# datascience
# how would we see the percent change by each age group?
percent_change = (us_pop.column('2025') - us_pop.column('2020')) / us_pop.column('2020')
us_pop_change = us_pop.with_column("Percent Change", percent_change).set_format("Percent Change", PercentFormatter)
us_pop_change.where('SEX', 0).where('AGE', are.above_or_equal_to(98)).sort('AGE').show()


In [ ]:
# polars
# how would we see the percent change by each age group?
percent_change_pl = (us_pop_pl.get_column('2025') - us_pop_pl.get_column('2020')) / us_pop_pl.get_column('2020')
us_pop_change_pl = us_pop_pl.with_columns(pl.Series("Percent Change", percent_change_pl))
with pl.Config(tbl_rows=-1):
    display(us_pop_change_pl.filter((pl.col('SEX') == (0))).filter((pl.col('AGE') >= (98))).sort('AGE'))


**DISCUSS**: Why are there so many more elderly in 2025?

### 2025 Sex Ratios

In [ ]:
# datascience
# focus just on 2025 population estimates, now by male/female
us_pop_2025 = us_pop.drop('2020')
us_pop_2025.show(3)


In [ ]:
# polars
# focus just on 2025 population estimates, now by male/female
us_pop_2025_pl = us_pop_pl.drop('2020')
us_pop_2025_pl.head(3)


In [ ]:
# datascience
all_ages = us_pop_2025.where('AGE', are.equal_to(999))
all_ages


In [ ]:
# polars
all_ages_pl = us_pop_2025_pl.filter((pl.col('AGE') == (999)))
all_ages_pl


In [ ]:
# datascience
infants = us_pop_2025.where('AGE', are.equal_to(0))
infants


In [ ]:
# polars
infants_pl = us_pop_2025_pl.filter((pl.col('AGE') == (0)))
infants_pl


In [ ]:
# datascience
old = us_pop.where('AGE', 100)
old


In [ ]:
# polars
old_pl = us_pop_pl.filter((pl.col('AGE') == (100)))
old_pl


**DISCUSS**: How does the ratio of females to males vary by age?

In [ ]:
# datascience
females_all_rows = us_pop_2025.where('SEX', are.equal_to(2))
females = females_all_rows.where('AGE', are.not_equal_to(999))
females.show(3)


In [ ]:
# polars
females_all_rows_pl = us_pop_2025_pl.filter((pl.col('SEX') == (2)))
females_pl = females_all_rows_pl.filter((pl.col('AGE') != (999)))
females_pl.head(3)


In [ ]:
# datascience
males_all_rows = us_pop_2025.where('SEX', are.equal_to(1))
males = males_all_rows.where('AGE', are.not_equal_to(999))
males.show(3)


In [ ]:
# polars
males_all_rows_pl = us_pop_2025_pl.filter((pl.col('SEX') == (1)))
males_pl = males_all_rows_pl.filter((pl.col('AGE') != (999)))
males_pl.head(3)


How does the ratio of females to males vary by age?

In [ ]:
# datascience
f_to_m_ratios = females.column(2) / males.column(2)

ratios = Table().with_columns(
    'Age', females.column('AGE'),
    'F:M Ratio', f_to_m_ratios
)

ratios


In [ ]:
# polars
f_to_m_ratios_pl = females_pl.to_series(2) / males_pl.to_series(2)

ratios_pl = pl.DataFrame([
    (females_pl.get_column('AGE')).alias('Age'),
    pl.Series('F:M Ratio', f_to_m_ratios_pl)
])

ratios_pl


In [ ]:
# datascience
ratios.sort('Age', descending=True)


In [ ]:
# polars
ratios_pl.sort('Age', descending=True)


#### Line Plot

In [ ]:
# datascience
ratios.plot('Age', 'F:M Ratio')


In [ ]:
# polars
# hvplot
ratios_pl.hvplot.line(x='Age', y='F:M Ratio')
